# 🧪 Praktikum 01 – Entwicklungsumgebung & erste LLM-Interaktion
**Applied Generative AI – NLP | Sommersemester 2026**

> ⏱️ **Gesamtdauer: ~90 Minuten**  
> 🎯 **Lernziele:** Python-Umgebung verifizieren · Ollama installieren & bedienen · lokale LLM-Inferenz via REST-API und Python-Library · Systemparameter erkunden

---

## Vorbereitung – Was du brauchst
| Tool | Version | Installationsquelle |
|------|---------|---------------------|
| Python | 3.11+ | python.org |
| Ollama | aktuell | [ollama.com](https://ollama.com) |
| Jupyter Lab/Notebook | aktuell | `pip install jupyterlab` |

**Terminal-Befehl zum Starten des Modells:**
```bash
ollama pull qwen2.5:7b
ollama serve          # Falls noch nicht gestartet
```


## Teil 1 – Umgebung prüfen ⏱️ ~10 min

In [ ]:
import sys, subprocess, importlib, platform

print("=" * 55)
print("  SYSTEMCHECK")
print("=" * 55)
print(f"Python      : {sys.version}")
print(f"Platform    : {platform.platform()}")
print()

required = ["requests", "ollama", "rich"]
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  ✅  {pkg:<12} installiert")
    except ImportError:
        print(f"  ❌  {pkg:<12} FEHLT  → pip install {pkg}")

print()
# Ollama-Prozess prüfen
try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print("Ollama-Status : ✅ erreichbar")
        print("Verfügbare Modelle:")
        print(result.stdout if result.stdout.strip() else "(keine Modelle gelistet)")
    else:
        print("Ollama-Status : ❌ nicht erreichbar")
        print(result.stderr.strip() or "Unbekannter Fehler bei 'ollama list'")
        print("→ Terminal: ollama serve")
except FileNotFoundError:
    print("Ollama-Status : ❌ ollama CLI nicht gefunden")
    print("→ Installiere Ollama und starte dann: ollama serve")
except Exception as e:
    print(f"Ollama-Status : ❌  {e}")
    print("→ Terminal: ollama serve")


In [ ]:
# Fehlende Pakete installieren (falls nötig)
# !pip install requests ollama rich


## Teil 2 – Ollama API direkt mit `requests` ⏱️ ~20 min

Ollama stellt unter `http://localhost:11434` eine REST-API bereit.  
Wir nutzen zunächst das rohe HTTP-Interface, um zu verstehen, wie LLM-Kommunikation *unter der Haube* funktioniert.

### 2.1 – Verfügbare Modelle abfragen

In [ ]:
import requests, json

BASE_URL = "http://localhost:11434"
MODEL    = "qwen2.5:7b"   # ← ggf. anpassen, z.B. "llama3.2:3b"

resp = requests.get(f"{BASE_URL}/api/tags")
resp.raise_for_status()

models = resp.json().get("models", [])
print(f"Lokale Modelle ({len(models)}):")
for m in models:
    size_gb = m.get("size", 0) / 1e9
    print(f"  • {m['name']:<30}  {size_gb:.1f} GB")


### 2.2 – Einfache Completion (non-streaming)

In [ ]:
payload = {
    "model": MODEL,
    "prompt": "Erkläre in einem Satz, was ein Transformer ist.",
    "stream": False,
    "options": {"temperature": 0.3}
}

resp = requests.post(f"{BASE_URL}/api/generate", json=payload)
resp.raise_for_status()

data = resp.json()
print("Antwort:", data["response"])
print()
print(f"Tokens erzeugt   : {data.get('eval_count', '?')}")
print(f"Tokens/Sek       : {data.get('eval_count', 0) / max(data.get('eval_duration', 1), 1) * 1e9:.1f}")
print(f"Gesamtdauer      : {data.get('total_duration', 0) / 1e9:.2f}s")


### 2.3 – Streaming-Antwort in Echtzeit

In [ ]:
import sys

payload_stream = {
    "model": MODEL,
    "prompt": "Nenne die 5 wichtigsten Meilensteine in der KI-Geschichte.",
    "stream": True,
    "options": {"temperature": 0.5}
}

print("Streaming-Antwort:\n" + "─" * 50)
with requests.post(f"{BASE_URL}/api/generate", json=payload_stream, stream=True) as r:
    r.raise_for_status()
    for line in r.iter_lines():
        if line:
            chunk = json.loads(line)
            print(chunk.get("response", ""), end="", flush=True)
            if chunk.get("done"):
                break
print("\n" + "─" * 50)


## Teil 3 – Chat-Interface mit der `ollama` Python-Library ⏱️ ~20 min

Die `ollama`-Library ist eine höhere Abstraktion über die REST-API  
und unterstützt **Multi-Turn-Conversations** mit Rollenverwaltung.

In [ ]:
import ollama

response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Was ist der Unterschied zwischen KI und Machine Learning?"}
    ]
)

print(response["message"]["content"])


### 3.1 – Mehrrundigem Gespräch führen

In [ ]:
conversation = [
    {"role": "system",  "content": "Du bist ein präziser NLP-Tutor. Antworte auf Deutsch, maximal 3 Sätze."},
    {"role": "user",    "content": "Was ist ein Large Language Model?"},
]

resp = ollama.chat(model=MODEL, messages=conversation)
assistant_reply = resp["message"]["content"]
conversation.append({"role": "assistant", "content": assistant_reply})

print("Assistent:", assistant_reply)

# Folge-Frage
follow_up = "Wie viele Parameter hat GPT-4 ungefähr?"
conversation.append({"role": "user", "content": follow_up})

resp2 = ollama.chat(model=MODEL, messages=conversation)
print("\nFollowup-Antwort:", resp2["message"]["content"])


## Teil 4 – Parameter erkunden ⏱️ ~25 min

### 4.1 – Temperature-Effekt

`temperature` kontrolliert die Zufälligkeit der Ausgabe:  
- `0.0` → meist sehr stabil / oft gleich, aber nicht in jeder Umgebung strikt identisch  
- `1.0` → kreativer / variabler  
- `> 1.5` → deutlich unvorhersehbarer

In [ ]:
prompt = "Erfinde einen kreativen Namen für ein KI-Startup."

print("Gleicher Prompt, verschiedene Temperaturen:\n")
for temp in [0.0, 0.5, 1.0, 1.5]:
    resp = ollama.generate(model=MODEL, prompt=prompt,
                           options={"temperature": temp, "seed": 42})
    print(f"  T={temp:.1f} → {resp['response'].strip()[:80]}")


### 4.2 – System-Prompt Einfluss demonstrieren

In [ ]:
user_question = "Erkläre mir Backpropagation."

personas = {
    "Grundschullehrer":
        "Du erklärst komplexe Themen so einfach, dass sie ein 10-jähriges Kind verstehen kann.",
    "Wissenschaftler":
        "Du gibst exakte, mathematisch präzise Antworten mit Fachbegriffen.",
    "Comedian":
        "Du erklärst alles mit Witzen und Analogien zum Alltag.",
}

for name, system_prompt in personas.items():
    resp = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_question},
        ],
        options={"temperature": 0.7}
    )
    text = resp["message"]["content"].strip().replace("\n", " ")
    print(f"[{name}]\n{text[:200]}...\n")


### 4.3 – Latenz-Benchmark ⏱️ ~10 min

In [ ]:
import time

test_prompts = [
    "Nenne die Hauptstadt von Deutschland.",           # kurz
    "Erkläre in 3 Sätzen, was ein neuronales Netz ist.",  # mittel
    "Schreibe eine 10-Punkte-Zusammenfassung der wichtigsten KI-Entwicklungen seit 2017.",  # lang
]

print(f"{'Prompt (gekürzt)':<55} {'Tokens':>7} {'Sek':>6} {'Tok/s':>7}")
print("─" * 80)
for p in test_prompts:
    t0 = time.time()
    r = ollama.generate(model=MODEL, prompt=p, options={"temperature": 0.0})
    elapsed = time.time() - t0
    tokens = r.get("eval_count", 0)
    print(f"{p[:52]:<55} {tokens:>7} {elapsed:>6.2f} {tokens/elapsed:>7.1f}")


## Teil 5 – Abschluss & Reflexion ⏱️ ~5 min

### ✅ Was du heute gelernt hast

| Konzept | Umsetzung |
|---------|-----------|
| Ollama API | REST-Calls mit `requests` |
| Streaming | Echtzeit-Ausgabe in Chunks |
| Chat-Memory | Multi-Turn mit Rollenverwaltung |
| Temperature | Einfluss auf Kreativität/Determinismus |
| System-Prompts | Persona-Steuerung |

### 🧩 Aufgaben zur Vertiefung

1. **Experimentiere mit einem anderen Modell** (z.B. `llama3.2:3b`) und vergleiche Qualität und Geschwindigkeit.
2. **Baue eine einfache interaktive Chat-Schleife** in Python, die auf Nutzereingaben wartet (`input()`).
3. **Finde heraus:** Ab welcher `temperature` werden die Antworten auf eine einfache Faktenfrage unzuverlässig?